<a href="https://colab.research.google.com/github/Altaieb-Mohammed/lab_2corse/blob/master/lab_10_las.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Если nltk не установлен, раскомментируйте и выполните:
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# 1. Загрузка данных
df = pd.read_csv('IMDB Dataset.csv')  # Убедитесь, что файл в рабочей директории
print(f"Всего отзывов: {len(df)}")
print(df.head())

# 2. Функция очистки текста
def clean_text(text):
    # Приводим к нижнему регистру
    text = text.lower()
    # Удаляем HTML-теги
    text = re.sub(r'<.*?>', ' ', text)
    # Удаляем все символы, кроме букв и пробелов
    text = re.sub(r'[^a-z\s]', '', text)
    # Удаляем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 3. Лемматизация и удаление стоп-слов
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = clean_text(text)
    words = text.split()
    # Удаляем стоп-слова и лемматизируем
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Применяем предобработку к отзывам (может занять время)
df['clean_review'] = df['review'].apply(preprocess_text)

# 4. Подготовка признаков и меток
X = df['clean_review']
y = df['sentiment'].map({'positive': 1, 'negative': 0})  # Преобразуем метки в 0 и 1

# 5. Векторизация текста с помощью TF-IDF
vectorizer = TfidfVectorizer()
X_vect = vectorizer.fit_transform(X)

# 6. Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_vect, y, test_size=0.2, random_state=42, stratify=y)

# 7. Обучение модели SVM
model = LinearSVC()
model.fit(X_train, y_train)

# 8. Оценка модели
y_pred = model.predict(X_test)
print("Отчёт по классификации:\n", classification_report(y_test, y_pred))
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}")

# 9. Выводы
print("""
Выводы:
- Предобработка текста (очистка, лемматизация, удаление стоп-слов) улучшает качество модели.
- Использование TF-IDF позволяет учитывать важность слов, что повышает точность классификации.
- Модель SVM показала хорошие результаты для задачи бинарной классификации отзывов.
""")
